In [ ]:
!nvidia-smi


In [ ]:
!pip install -q transformers accelerate bitsandbytes datasets

In [ ]:
import torch, time, re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

name = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
tok.padding_side = "left"

In [ ]:
ids = tok("What is the capital of France?")["input_ids"]
print(ids)
print(tok.convert_ids_to_tokens(ids))
print(tok.decode(ids))

ids = tok("Quantization of transformers is unbelievably useful")["input_ids"]
print(ids)
print(tok.convert_ids_to_tokens(ids))

In [ ]:
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16).to("cuda")

In [ ]:
!nvidia-smi

In [ ]:
def bench(model, device, prompts, max_new_tokens=64, runs=3):
    inputs = tok(prompts, return_tensors="pt", padding=True).to(device)
    gen = dict(max_new_tokens=max_new_tokens, min_new_tokens=max_new_tokens, do_sample=False)
    model.generate(**inputs, **{**gen, "max_new_tokens": 8, "min_new_tokens": 8})  # warmup
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    times = []
    for _ in range(runs):
        if device == "cuda":
            torch.cuda.synchronize()
            s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
            s.record()
            out = model.generate(**inputs, **gen)
            e.record(); torch.cuda.synchronize()
            times.append(s.elapsed_time(e) / 1000)
        else:
            t0 = time.perf_counter()
            out = model.generate(**inputs, **gen)
            times.append(time.perf_counter() - t0)
    avg = sum(times) / len(times)
    new_tokens = max_new_tokens * len(prompts)
    mem = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else None
    return {"latency_s": avg, "tokens_per_s": new_tokens / avg, "peak_mem_gb": mem}


In [ ]:
test_prompts = ["What is the capital of France?"]
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16).to("cuda")
print(bench(model, "cuda", test_prompts, max_new_tokens=32))

In [ ]:
print(bench(model, "cuda", test_prompts, max_new_tokens=32))
print(bench(model, "cuda", test_prompts, max_new_tokens=32))

Baseline, batch size 1, fp16, 32 new tokens, Tesla T4
Session 1: 2.31 s (13.9 tok/s, slow first call), 1.24 s (25.8), 1.26 s (25.4)
Session 2: 1.27 s (25.2), 1.28 s (25.0)
Peak memory: 3.10 GB
Note: the first call in session 1 was ~2x slower and didn't reproduce, probably a one-time cold start.

In [ ]:
prompts = [
    "What is the capital of France?",
    "Explain what a GPU is in two sentences.",
    "Write a Python function that returns the nth Fibonacci number.",
    "List three differences between a CPU and a GPU.",
    "If a train travels 60 miles in 1.5 hours, what is its average speed?",
    "Translate to Spanish: Where is the nearest train station?",
    "Give me three tips for preparing for a technical interview.",
    "What does the word 'latency' mean in computing?",
    "Write a haiku about the ocean.",
    "Explain what quantization means for a neural network in simple terms.",
    "What is 17 multiplied by 23?",
    "Write a short, polite email asking a professor for an extension.",
    "Name three uses of machine learning in healthcare.",
    "What is the difference between a list and a tuple in Python?",
    "Describe how a CUDA thread block works in one paragraph.",
    "Why is the sky blue?",
]
print(len(prompts))

enc = tok(prompts[:4], return_tensors="pt", padding=True)
print(enc["input_ids"].shape)
print(enc["attention_mask"])

In [ ]:
import gc
del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!nvidia-smi

In [ ]:
cpu_model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)  # stays on CPU
print("CPU threads:", torch.get_num_threads())
cpu_result = bench(cpu_model, "cpu", prompts[:1], max_new_tokens=32, runs=2)
print("CPU fp32:", cpu_result)
del cpu_model
gc.collect()

In [ ]:
gpu_model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).to("cuda")
gpu_result = bench(gpu_model, "cuda", prompts[:1], max_new_tokens=32, runs=3)
print("GPU fp32:", gpu_result)

In [ ]:
print("GPU fp32 vs CPU fp32 speedup:", gpu_result["tokens_per_s"] / cpu_result["tokens_per_s"])

In [ ]:
del gpu_model
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16).to("cuda")

In [ ]:
from torch.profiler import profile, ProfilerActivity

inputs = tok(prompts[:1], return_tensors="pt").to("cuda")

for _ in range(2):
    model.generate(**inputs, max_new_tokens=8, do_sample=False)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
    model.generate(**inputs, max_new_tokens=16, do_sample=False)

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

In [ ]:
results = []
for bs in [1, 4, 16]:
    r = bench(model, "cuda", prompts[:bs], max_new_tokens=32, runs=3)
    r["batch"] = bs
    results.append(r)
    print(bs, r)

Batch	Latency (s)	Tokens/s	vs batch 1	Peak mem (GB)
1	1.62	19.8	1x	3.104
4	1.50	85.5	4.3x	3.119
16	1.56	329.1	16.6x	3.177

In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
print("still defined:", [n for n in ["model", "gpu_model", "cpu_model"] if n in globals()])
print("missing setup:", [n for n in ["tok", "bench", "prompts", "name"] if n not in globals()])

In [ ]:
for n in ["model", "gpu_model", "cpu_model", "prof", "inputs"]:
    if n in globals():
        del globals()[n]
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!nvidia-smi

In [ ]:
def load(mode):
    if mode == "fp32":
        return AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).to("cuda")
    if mode == "fp16":
        return AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16).to("cuda")
    if mode == "int8":
        cfg = BitsAndBytesConfig(load_in_8bit=True)
        return AutoModelForCausalLM.from_pretrained(name, quantization_config=cfg, device_map="cuda")
    if mode == "int4":
        cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        return AutoModelForCausalLM.from_pretrained(name, quantization_config=cfg, device_map="cuda")

In [ ]:
import pandas as pd

rows = []
for mode in ["fp32", "fp16", "int8", "int4"]:
    m = load(mode)
    for bs in [1, 4, 16]:
        r = bench(m, "cuda", prompts[:bs], max_new_tokens=32, runs=3)
        r.update({"mode": mode, "batch": bs})
        rows.append(r)
        print(mode, bs, round(r["tokens_per_s"], 1), "tok/s,", round(r["peak_mem_gb"], 2), "GB")
    del m
    gc.collect()
    torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df.to_csv("precision_results.csv", index=False)
df

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for mode, g in df.groupby("mode", sort=False):
    ax[0].plot(g["batch"], g["tokens_per_s"], marker="o", label=mode)
ax[0].set_xlabel("batch size"); ax[0].set_ylabel("tokens / second")
ax[0].set_title("Throughput vs batch size"); ax[0].legend()

mem = df.groupby("mode", sort=False)["peak_mem_gb"].max()
ax[1].bar(mem.index, mem.values)
ax[1].set_ylabel("peak GPU memory (GB)"); ax[1].set_title("Memory by precision")
plt.tight_layout(); plt.savefig("precision_charts.png", dpi=150)
print("saved")

In [ ]:
from numba import cuda
import numpy as np, math

@cuda.jit
def vec_add(a, b, c):
    i = cuda.grid(1)      # this thread's global index
    if i < c.size:        # extra threads from rounding up must not write out of bounds
        c[i] = a[i] + b[i]

n = 10_000_000
a = np.random.rand(n).astype(np.float32)
b = np.random.rand(n).astype(np.float32)

d_a, d_b = cuda.to_device(a), cuda.to_device(b)    # CPU -> GPU memory
d_c = cuda.device_array(n, dtype=np.float32)       # empty output on the GPU

threads = 256
blocks = math.ceil(n / threads)                    # round up to cover all n
vec_add[blocks, threads](d_a, d_b, d_c)
cuda.synchronize()                                 # launch is async, so wait

print(np.allclose(d_c.copy_to_host(), a + b))

In [ ]:
from numba import cuda
import numpy as np, math, time, torch

@cuda.jit
def matmul_naive(A, B, C):
    col, row = cuda.grid(2)           # x = column, y = row; x varies fastest across threads
    if row < C.shape[0] and col < C.shape[1]:
        s = 0.0
        for k in range(A.shape[1]):   # dot product of one row of A and one column of B
            s += A[row, k] * B[k, col]
        C[row, col] = s

N = 1024
A = np.random.rand(N, N).astype(np.float32)
B = np.random.rand(N, N).astype(np.float32)

dA, dB = cuda.to_device(A), cuda.to_device(B)
dC = cuda.device_array((N, N), dtype=np.float32)

tpb = (16, 16)                                    # 256 threads per block
bpg = (math.ceil(N / 16), math.ceil(N / 16))      # blocks per grid, 2D

matmul_naive[bpg, tpb](dA, dB, dC)                # warmup: first call includes JIT compile time
cuda.synchronize()

# time my kernel
t0 = time.perf_counter()
for _ in range(10):
    matmul_naive[bpg, tpb](dA, dB, dC)
cuda.synchronize()                                # wait before stopping the clock
mine_ms = (time.perf_counter() - t0) / 10 * 1000

# time PyTorch's matmul (cuBLAS underneath) on the same data
tA, tB = torch.from_numpy(A).cuda(), torch.from_numpy(B).cuda()
torch.matmul(tA, tB)                              # warmup
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(10):
    tC = torch.matmul(tA, tB)
torch.cuda.synchronize()
torch_ms = (time.perf_counter() - t0) / 10 * 1000

# correctness first, then speed
print("correct:", np.allclose(dC.copy_to_host(), tC.cpu().numpy(), rtol=1e-3, atol=1e-3))
print(f"my kernel: {mine_ms:.2f} ms | torch/cuBLAS: {torch_ms:.2f} ms | ratio: {mine_ms / torch_ms:.1f}x slower")

In [ ]:
from numba import cuda, float32

TPB = 16   # tile size = threads per block side

@cuda.jit
def matmul_tiled(A, B, C):
    sA = cuda.shared.array((TPB, TPB), dtype=float32)   # fast memory shared by the block
    sB = cuda.shared.array((TPB, TPB), dtype=float32)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    col, row = cuda.grid(2)
    acc = 0.0
    for t in range((A.shape[1] + TPB - 1) // TPB):      # walk across the tiles
        # each thread loads one element of each tile (0 if out of bounds)
        sA[ty, tx] = A[row, t*TPB + tx] if row < A.shape[0] and t*TPB + tx < A.shape[1] else 0.0
        sB[ty, tx] = B[t*TPB + ty, col] if col < B.shape[1] and t*TPB + ty < B.shape[0] else 0.0
        cuda.syncthreads()                              # wait until the whole tile is loaded
        for k in range(TPB):
            acc += sA[ty, k] * sB[k, tx]
        cuda.syncthreads()                              # wait before the tile is overwritten
    if row < C.shape[0] and col < C.shape[1]:
        C[row, col] = acc

# same launch shape as before; warmup, then time it
dC2 = cuda.device_array((N, N), dtype=np.float32)
matmul_tiled[bpg, tpb](dA, dB, dC2); cuda.synchronize()
t0 = time.perf_counter()
for _ in range(10):
    matmul_tiled[bpg, tpb](dA, dB, dC2)
cuda.synchronize()
tiled_ms = (time.perf_counter() - t0) / 10 * 1000

print("correct:", np.allclose(dC2.copy_to_host(), tC.cpu().numpy(), rtol=1e-3, atol=1e-3))
print(f"tiled: {tiled_ms:.2f} ms | vs naive: {mine_ms / tiled_ms:.1f}x faster | vs cuBLAS: {tiled_ms / torch_ms:.1f}x slower")

In [ ]:
def time_kernel(k, out, reps=20):
    k[bpg, tpb](dA, dB, out); cuda.synchronize()    # warmup
    t0 = time.perf_counter()
    for _ in range(reps):
        k[bpg, tpb](dA, dB, out)
    cuda.synchronize()
    return (time.perf_counter() - t0) / reps * 1000

for run in range(6):
    print(f"naive {time_kernel(matmul_naive, dC):.2f} ms | tiled {time_kernel(matmul_tiled, dC2):.2f} ms")

In [ ]:
!nvidia-smi -q -d CLOCK,TEMPERATURE

| Kernel | Time | vs cuBLAS |
| :--- | :--- | :--- |
| **Naive** | ~22.9 ms | ~28x slower |
| **Tiled (16x16)** | ~21.9 ms | ~26x slower |
| **cuBLAS** | ~0.83 ms | 1x |

   GSM8K evaluation. The first fp16 run used 256 new tokens and a looser extract() that fell back to the last number in the reply; 25 of 50 replies were unfinished. I switched to 512 tokens and strict scoring (first number after "Answer:") and re-ran fp16, then ran the other three precisions with the same settings. The numbers in the README come from the 512-token runs.

In [ ]:
from datasets import load_dataset
import re

# first 50 test questions of GSM8K (grade-school math word problems)
ds = load_dataset("openai/gsm8k", "main", split="test").select(range(50))

def gold(ans):
    # reference answers end with "#### <number>", so take what follows the marker
    return ans.split("####")[-1].strip().replace(",", "")

def extract(text):
    # pull the last number after "Answer:" from the model's reply, or None if there isn't one
    tail = text.split("Answer:")[-1]
    nums = re.findall(r"-?\d[\d,]*\.?\d*", tail)
    return nums[-1].replace(",", "").rstrip(".") if nums else None

print(ds[0]["question"])
print("gold:", gold(ds[0]["answer"]))
print("extract test:", extract("so she has 12 apples. Answer: 1,234.5"))   # expect 1234.5

In [ ]:
import time

def make_prompt(q):
    # chat format the Qwen instruct model expects, with a fixed answer format so extract() works
    msgs = [{"role": "user", "content": f"{q}\nSolve step by step. End with 'Answer: <number>'."}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def run_eval(model, n=50, bs=8, max_new=256):
    results = []
    for i in range(0, n, bs):
        batch = [ds[j] for j in range(i, min(i + bs, n))]
        prompts_b = [make_prompt(x["question"]) for x in batch]
        enc = tok(prompts_b, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new, do_sample=False)   # greedy: repeatable
        # drop the prompt tokens, keep only the newly generated text
        texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        for x, t in zip(batch, texts):
            pred, ref = extract(t), gold(x["answer"])
            results.append({"pred": pred, "gold": ref, "correct": pred == ref, "text": t})
    return results

m = load("fp16")
t0 = time.time()
res = run_eval(m)
print(f"fp16: {sum(r['correct'] for r in res)}/{len(res)} correct in {time.time() - t0:.0f}s")
print("no 'Answer:' found:", sum(r["pred"] is None for r in res))

In [ ]:
# show 3 wrong answers so we can tell real math errors from formatting mismatches
for r in [r for r in res if not r["correct"]][:3]:
    print("gold:", r["gold"], "| pred:", r["pred"])
    print(r["text"][:300])
    print("---")

In [ ]:
no_answer = sum("Answer:" not in r["text"] for r in res)
print("replies without 'Answer:':", no_answer, "of", len(res))

In [ ]:
def extract(text):
    # strict: only trust a number that follows "Answer:", otherwise None
    if "Answer:" not in text:
        return None
    nums = re.findall(r"-?\d[\d,]*\.?\d*", text.split("Answer:")[-1])
    return nums[0].replace(",", "").rstrip(".") if nums else None   # first number after the marker

In [ ]:
m = load("fp16")
t0 = time.time()
res = run_eval(m, max_new=512)
ok = sum(r["correct"] for r in res)
unfinished = sum(r["pred"] is None for r in res)
print(f"fp16 @512: {ok}/50 correct | unfinished: {unfinished} | {time.time() - t0:.0f}s")

In [ ]:
print("missing:", [n for n in ["m", "res", "load", "run_eval", "make_prompt", "extract", "gold", "ds", "tok", "time", "json"] if n not in globals()])

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!nvidia-smi

In [ ]:
import json, os, logging
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

OUT = "gsm8k_results.json"
all_res = json.load(open(OUT)) if os.path.exists(OUT) else {}
print("already saved:", list(all_res))                 # should show ['fp16']

for mode in ["fp32", "int8", "int4"]:
    if mode in all_res:
        continue                                       # resume after a crash
    m = load(mode)
    t0 = time.time()
    all_res[mode] = run_eval(m, max_new=512)
    json.dump(all_res, open(OUT, "w"))                 # save right after each mode
    r = all_res[mode]
    print(f"{mode}: {sum(x['correct'] for x in r)}/50 correct | "
          f"unfinished: {sum(x['pred'] is None for x in r)} | {time.time() - t0:.0f}s")
    del m; gc.collect(); torch.cuda.empty_cache()

In [ ]:
for mode in ["int8", "int4"]:
    wrong = [r for r in all_res[mode] if not r["correct"]][:3]
    print("=====", mode)
    for r in wrong:
        print("gold:", r["gold"], "| pred:", r["pred"]); print(r["text"][:250]); print("---")

In [ ]:
fp = all_res["fp16"]
for mode in ["int8", "int4"]:
    lost = [i for i in range(50) if fp[i]["correct"] and not all_res[mode][i]["correct"]]
    gained = [i for i in range(50) if not fp[i]["correct"] and all_res[mode][i]["correct"]]
    print(mode, "| right in fp16, wrong here:", len(lost), "| wrong in fp16, right here:", len(gained))

In [3]:
import json, os, urllib.request
import pandas as pd

# raw URL base for files committed to the repo's main branch
BASE = "https://raw.githubusercontent.com/assen-v/llm-gpu-benchmark/main/"

def src(name):
    # use the local file if this session has it, otherwise read the committed copy
    return name if os.path.exists(name) else BASE + name

print("Speed and memory")
display(pd.read_csv(src("precision_results.csv")).round(2))

# the JSON needs an explicit open: local path or URL
path = src("gsm8k_results.json")
all_res = json.load(open(path)) if os.path.exists(path) else json.load(urllib.request.urlopen(path))

rows = [{"precision": m,
         "correct_of_50": sum(x["correct"] for x in all_res[m]),
         "unfinished": sum(x["pred"] is None for x in all_res[m])}
        for m in ["fp32", "fp16", "int8", "int4"]]
print("\nGSM8K accuracy (50 questions, 512 new tokens)")
display(pd.DataFrame(rows))

Speed and memory


,latency_s,tokens_per_s,peak_mem_gb,mode,batch
0,1.42,22.59,6.19,fp32,1
1,1.47,86.79,6.21,fp32,4
2,2.11,242.81,6.29,fp32,16
3,1.54,20.72,3.10,fp16,1
4,1.46,87.93,3.12,fp16,4
5,1.51,339.33,3.18,fp16,16
6,14.35,2.23,1.80,int8,1
7,15.02,8.52,1.82,int8,4
8,15.02,34.09,1.87,int8,16
9,2.68,11.95,1.23,int4,1



GSM8K accuracy (50 questions, 512 new tokens)


,precision,correct_of_50,unfinished
0,fp32,31,8
1,fp16,32,8
2,int8,24,10
3,int4,23,5
